# TUE Reimbursement

This notebook demonstrates goal-oriented alignment visualization for the TUE Reimbursment case study.

- Models are loaded in the notebook.
- Handmade test cases are defined inline based on the traces discussed in the paper.



## Setup & Imports

In [14]:
import pandas as pd

import pm4py
from pm4py.objects.conversion.log import converter as log_converter
from pm4py.objects.log.obj import EventLog


from Semantics.goccva_pipeline import analyse

from Ui.goccva_ui import render_from_analysis, render_all_goal_oriented_alignments_from_analysis, render_case_distribution_matrix

from Semantics.goccva_helpers import sequences_to_event_log

from Semantics.istar_processor import read_istar_model
from Semantics.petri_net_processor import read_petri_net
from Semantics.event_mapping_from_csv import read_event_mapping_csv

from Ui.interface import InterfaceBuilder, WhatIfInterfaceBuilder

from pprint import pp

## Load Models

Load the goal model, process model, and mapping used by the case study.

In [15]:

# Paths used in the GoCCvA repository
goal_model_path = "content/TUEReimbursement/GMjcavi2.txt"
process_model_path = "content/TUEReimbursement/domestic_declaration_ilpn_updated.pnml"
mapping_path = "content/TUEReimbursement/mappingjcavi.csv"

goal_model = read_istar_model(str(goal_model_path), qualified=True)

petri_net = read_petri_net(str(process_model_path))

activity_mapping = read_event_mapping_csv(str(mapping_path))

# Fix minor inconsistencies in the mapping vs. the goal model (e.g., extra spaces, missing/extra parentheses, etc.)
# This should not be required as I have changed GMjcavi2.txt to make it compatiible with the mapping.
activity_mapping = goal_model.canonicalize_activity_mapping(activity_mapping) 


## Kogi

In [16]:
# Kogi needs a mapping from transition names to intentional elements
# and not transition labels to intentional elements.
kogi_mapping = petri_net.convert_to_kogi_mapping(activity_mapping)


## What If Scenario

In [17]:
interface = WhatIfInterfaceBuilder(goal_model).create_interface()
display(interface)

In [18]:
interface = InterfaceBuilder(goal_model,petri_net=petri_net,event_mapping=kogi_mapping).create_interface()

display(interface)

## Target Configuration

Define the target requirements to be evaluated. The make, break, and non-related sets are computed from the loaded goal model.

In [19]:
from Semantics.target_sets import compute_target_sets, target_sets_as_rows


targets = [
    '(Admin) adequate declaration handling',
    '(Employee) Increase employee satisfaction',
]

pp(target_sets_as_rows(compute_target_sets(goal_model, targets)))

[{'target': '(Admin) adequate declaration handling',
  'MakeSet': '(Admin) Approve Declaration, (Admin) Handle payment, (Budget '
             'Owner) Approve Declaration, (Supervisor) Approve Declaration, '
             '(Supervisor) Request Payment',
  'BreakSet': '',
  'NRSet': '(Admin) Reject Declaration, (Budget Owner) Reject Declaration, '
           '(Employee) Reject Declaration, (Employee) Save Declaration, '
           '(Employee) Submit Declaration, (Supervisor) Reject Declaration'},
 {'target': '(Employee) Increase employee satisfaction',
  'MakeSet': '(Admin) Handle payment',
  'BreakSet': '(Admin) Reject Declaration, (Budget Owner) Reject Declaration, '
              '(Employee) Reject Declaration, (Supervisor) Reject Declaration',
  'NRSet': '(Admin) Approve Declaration, (Budget Owner) Approve Declaration, '
           '(Employee) Save Declaration, (Employee) Submit Declaration, '
           '(Supervisor) Approve Declaration, (Supervisor) Request Payment'}]


## Handmade Test Cases

Define the traces based on the example from the paper.

In [20]:
handmade_cases = [
    {
        "requested_case": "Alignment 1",
        "trace": [
            "Declaration SUBMITTED by EMPLOYEE",
            "Declaration REJECTED by ADMINISTRATION",
            "t_tau_rev",
            "Declaration SUBMITTED by EMPLOYEE",
            "Declaration APPROVED by ADMINISTRATION",
            "Declaration APPROVED by BUDGET OWNER",
            "Declaration FINAL_APPROVED by SUPERVISOR",
            "Request Payment",
            "Payment Handled",
        ],
        "why": "ds ra ds aa ba as rp ph",
    },
    {
        "requested_case": "Alignment 2",
        "trace": [
            "Declaration SUBMITTED by EMPLOYEE",
            "Declaration APPROVED by ADMINISTRATION",
            "Declaration APPROVED by BUDGET OWNER",
            "Declaration REJECTED by BUDGET OWNER",
            "Declaration FINAL_APPROVED by SUPERVISOR",
            "Request Payment",
            "Payment Handled",
        ],
        "why": "ds aa ba rb as rp ph",
    },
    {
        "requested_case": "Alignment 3",
        "trace": [
            "Declaration SUBMITTED by EMPLOYEE",
            "Declaration APPROVED by ADMINISTRATION",
            "Declaration APPROVED by BUDGET OWNER",
            "Request Payment",
            "Payment Handled",
        ],
        "why": "ds aa ba rp ph",
    },

]

print(f"Defined {len(handmade_cases)} test case")


Defined 3 test case


In [21]:
def handmade_cases_to_event_log(cases):
    sequences = []
    for case in cases:
        trace = case.get("trace")
        sequences.append(trace)
    return sequences_to_event_log(sequences)

handmade_event_log = handmade_cases_to_event_log(handmade_cases)
print(f"EventLog created with {len(handmade_event_log)} trace(s)")

EventLog created with 3 trace(s)


In [22]:
from Semantics.enums import ElementStatus

event_log = handmade_event_log

initial_marking = {
    "(Employee) Increase employee satisfaction": ElementStatus.SATISFIED,
}

summary, detailed, contribution_to_targets = analyse(
    goal_model,
    petri_net,
    event_log,
    targets,
    activity_mapping,
    initial_marking=initial_marking,
 )

print("\nContribution to targets")
pp(contribution_to_targets)

aligning log, completed variants ::   0%|          | 0/3 [00:00<?, ?it/s]


Contribution to targets
[{'target': '(Admin) adequate declaration handling',
  'MakeSet': '(Admin) Approve Declaration, (Admin) Handle payment, (Budget '
             'Owner) Approve Declaration, (Supervisor) Approve Declaration, '
             '(Supervisor) Request Payment',
  'BreakSet': '',
  'NRSet': '(Admin) Reject Declaration, (Budget Owner) Reject Declaration, '
           '(Employee) Reject Declaration, (Employee) Save Declaration, '
           '(Employee) Submit Declaration, (Supervisor) Reject Declaration'},
 {'target': '(Employee) Increase employee satisfaction',
  'MakeSet': '(Admin) Handle payment',
  'BreakSet': '(Admin) Reject Declaration, (Budget Owner) Reject Declaration, '
              '(Employee) Reject Declaration, (Supervisor) Reject Declaration',
  'NRSet': '(Admin) Approve Declaration, (Budget Owner) Approve Declaration, '
           '(Employee) Save Declaration, (Employee) Submit Declaration, '
           '(Supervisor) Approve Declaration, (Supervisor) Request

## Activity Abbreviations

In [23]:
activity_abbreviations = {
    "Declaration REJECTED by ADMINISTRATION": "ra",
    "Payment Handled": "ph",
    "Declaration REJECTED by BUDGET OWNER": "rb",
    "Declaration SAVED by EMPLOYEE": "dsv",
    "Declaration APPROVED by ADMINISTRATION": "aa",
    "Declaration REJECTED by EMPLOYEE": "er",
    "Request Payment": "rp",
    "Declaration SUBMITTED by EMPLOYEE": "ds",
    "Declaration APPROVED by BUDGET OWNER": "ba",
    "Declaration FINAL_APPROVED by SUPERVISOR": "as",
    "Declaration REJECTED by SUPERVISOR": "rs",
    "Declaration APPROVED by PRE_APPROVER": "pa",
    "Declaration REJECTED by PRE_APPROVER": "rpa",
    "Declaration REJECTED by MISSING": "rm",
    "t_tau_rev": "tau",
}

print("Activity abbreviations configured")


Activity abbreviations configured


## Render Visualization

In [24]:
render_all_goal_oriented_alignments_from_analysis(
    cases=handmade_cases,
    activity_abbreviations=activity_abbreviations,
    summary=summary,
    detailed=detailed,
    contribution_to_targets=contribution_to_targets,
    title="Goal-oriented Process Alignment Examples (GoCCvA Case Study)",
)

# Goal-oriented Process Alignment Examples (GoCCvA Case Study)

The following cases illustrate combinations of alignment class and target-fulfilment class. Each table separates the process alignment row from the target-specific interpretation rows.

# Reading the .xes file


In [25]:
log_file_path = "content/TUEReimbursement/DomesticDeclarations.xes.gz"

full_log = log_converter.apply(pm4py.read_xes(str(log_file_path)), variant=log_converter.Variants.TO_EVENT_LOG)

print("Log file loaded")
print(len(full_log))

parsing log, completed traces ::   0%|          | 0/10357 [00:00<?, ?it/s]

Log file loaded
10357


In [26]:
summary, detailed, contribution_to_targets = analyse(
    goal_model,
    petri_net,
    full_log,
    targets,
    activity_mapping,
    initial_marking=None,
 )

matrix_result = render_case_distribution_matrix(
    summary=summary,
    title="TUE Reimbursement Case Distribution Matrix",
    targets=targets,
)

print(matrix_result["counts"])

aligning log, completed variants ::   0%|          | 0/90 [00:00<?, ?it/s]

{'O+': 2411, 'O~': 0, 'O-': 185, 'N+': 1, 'N~': 314, 'N-': 7446}


### Show the result of the analysis of unique traces

In [27]:

traces = [[event['concept:name'] for event in trace] for trace in full_log]

from collections import Counter

traces_by_frequency = Counter(tuple(item) for item in traces)

unique_traces = [list(item) for item, _ in traces_by_frequency.most_common()]

summary, detailed, contribution_to_targets = analyse(
    goal_model,
    petri_net,
    sequences_to_event_log(unique_traces),
    targets,
    activity_mapping,
)

matrix_result = render_case_distribution_matrix(
    summary=summary,
    title="TUE Reimbursement Case Distribution Matrix",
    targets=targets,
)

print(matrix_result["counts"])


aligning log, completed variants ::   0%|          | 0/90 [00:00<?, ?it/s]

{'O+': 1, 'O~': 0, 'O-': 3, 'N+': 1, 'N~': 19, 'N-': 66}


## Show the result of the most used unique traces

In [28]:

most_used_traces_by_frequency = traces_by_frequency.most_common(2)

for index, (trace, count) in enumerate(most_used_traces_by_frequency):
    print(f"Case: {index+1}: {count} traces")

most_used_traces_log = sequences_to_event_log([list(item) for item, _ in most_used_traces_by_frequency])
                     
summary, detailed, contribution_to_targets = analyse(
    goal_model,
    petri_net,
    most_used_traces_log,
    targets,
    activity_mapping,
)

render_from_analysis(
    activity_abbreviations=activity_abbreviations,
    summary=summary,
    detailed=detailed,
    contribution_to_targets=contribution_to_targets,
    title="Goal-oriented Process Alignment Examples (GoCCvA Case Study)",
)


Case: 1: 4566 traces
Case: 2: 2411 traces


aligning log, completed variants ::   0%|          | 0/2 [00:00<?, ?it/s]

# Goal-oriented Process Alignment Examples (GoCCvA Case Study)

The following cases illustrate combinations of alignment class and target-fulfilment class. Each table separates the process alignment row from the target-specific interpretation rows.

In [31]:
targets = ["(Employee) Increase employee satisfaction"]

summary, detailed, contribution_to_targets = analyse(
    goal_model,
    petri_net,
    sequences_to_event_log(unique_traces),
    targets,
    activity_mapping,
)

matrix_result = render_case_distribution_matrix(
    summary=summary,
    title="Increase employee satisfaction",
    targets=targets,
)

print(matrix_result["counts"])

summary_df = pd.DataFrame(summary)
display(summary_df)

pp([s for s in summary if s["trace_id"] == 44])

pp([s for s in summary if s["trace_id"] == 87])


aligning log, completed variants ::   0%|          | 0/90 [00:00<?, ?it/s]

{'O+': 1, 'O~': 0, 'O-': 3, 'N+': 10, 'N~': 50, 'N-': 26}


,trace_id,trace,alignment_cost,fitness,traditional_class,goal_class
0,2,Declaration SUBMITTED by EMPLOYEE | Declaratio...,0,1.000000,optimal,Strongly compliant
1,7,Declaration SUBMITTED by EMPLOYEE | Declaratio...,0,1.000000,optimal,Non-compliant
2,25,Declaration SUBMITTED by EMPLOYEE | Declaratio...,0,1.000000,optimal,Non-compliant
3,39,Declaration SUBMITTED by EMPLOYEE | Declaratio...,0,1.000000,optimal,Non-compliant
4,1,Declaration SUBMITTED by EMPLOYEE | Declaratio...,10000,0.875000,non-optimal,Strongly compliant
...,...,...,...,...,...,...
85,76,Declaration SUBMITTED by EMPLOYEE | Declaratio...,20000,0.777778,non-optimal,Non-compliant
86,77,Declaration SUBMITTED by EMPLOYEE | Declaratio...,20000,0.818182,non-optimal,Non-compliant
87,80,Declaration SUBMITTED by EMPLOYEE | Declaratio...,30000,0.700000,non-optimal,Non-compliant
88,81,Declaration SUBMITTED by EMPLOYEE | Declaratio...,50000,0.687500,non-optimal,Non-compliant


[{'trace_id': 44,
  'trace': 'Declaration SUBMITTED by EMPLOYEE | Declaration APPROVED by '
           'PRE_APPROVER | Declaration FINAL_APPROVED by SUPERVISOR | '
           'Declaration REJECTED by MISSING',
  'alignment_cost': 50000,
  'fitness': 0.2857142857142857,
  'traditional_class': 'non-optimal',
  'goal_class': 'Non-compliant'}]
[{'trace_id': 87,
  'trace': 'Declaration SUBMITTED by EMPLOYEE | Declaration REJECTED by '
           'ADMINISTRATION | Declaration SUBMITTED by EMPLOYEE | Declaration '
           'REJECTED by ADMINISTRATION | Declaration SUBMITTED by EMPLOYEE | '
           'Declaration REJECTED by ADMINISTRATION | Declaration REJECTED by '
           'EMPLOYEE',
  'alignment_cost': 20000,
  'fitness': 0.8,
  'traditional_class': 'non-optimal',
  'goal_class': 'Non-compliant'}]
